<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/earlier_version/kNN_modelling_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SECTION 1: Setup — load both semi-raw datasets, configure
# cross-validation separately for each (different row counts
# due to different duplicate rates)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/'

df_2cluster = pd.read_csv(BASE_PATH + 'networkTraffic_knn_semiraw.csv')
df_7cluster = pd.read_csv(BASE_PATH + 'networkTraffic_knn_semiraw_7cluster.csv')

X_2c = df_2cluster.drop(columns=['attack_cat'])
y_2c = df_2cluster['attack_cat']
skf_2c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_7c = df_7cluster.drop(columns=['attack_cat'])
y_7c = df_7cluster['attack_cat']
skf_7c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("2-cluster branch:", X_2c.shape)
print("7-cluster branch:", X_7c.shape)

Mounted at /content/drive
2-cluster branch: (150243, 34)
7-cluster branch: (162745, 39)


In [ ]:
# ============================================================
# SECTION 2: Build the leak-safe preprocessing pipeline for
# the 2-cluster branch — scale, bucket, encode. NO CLAMPING,
# per the significant Section-10 finding that clamping the
# finally-selected features hurt performance (p=0.0075).
# ============================================================
nominal_cols = ['proto', 'state', 'service']

fold_data_full_2c = []
for fold_num, (train_idx, test_idx) in enumerate(skf_2c.split(X_2c, y_2c)):
    X_train, X_test = X_2c.iloc[train_idx].copy(), X_2c.iloc[test_idx].copy()
    y_train, y_test = y_2c.iloc[train_idx], y_2c.iloc[test_idx]

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    proto_counts = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full_2c.append((X_train_enc, X_test_enc, y_train, y_test))
    print(f"Fold {fold_num+1}: prepared. X_train: {X_train_enc.shape}, kept protos: {keep_protos}")

Fold 1: prepared. X_train: (120194, 56), kept protos: ['tcp', 'udp']
Fold 2: prepared. X_train: (120194, 57), kept protos: ['tcp', 'udp']
Fold 3: prepared. X_train: (120194, 57), kept protos: ['tcp', 'udp']
Fold 4: prepared. X_train: (120195, 55), kept protos: ['tcp', 'udp']
Fold 5: prepared. X_train: (120195, 57), kept protos: ['tcp', 'udp']


In [ ]:
# ============================================================
# SECTION 3: Baseline and distance-weighted voting (2-cluster
# branch, unclamped)
# ============================================================
f1_baseline = [f1_score(y_test, KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean').fit(X_train, y_train).predict(X_test), average='macro')
               for X_train, X_test, y_train, y_test in fold_data_full_2c]
print(f"Baseline (k=5, euclidean, uniform): {np.mean(f1_baseline):.4f} (± {np.std(f1_baseline):.4f})")

f1_weighted = [f1_score(y_test, KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean').fit(X_train, y_train).predict(X_test), average='macro')
               for X_train, X_test, y_train, y_test in fold_data_full_2c]
print(f"Distance-weighted (k=5, euclidean): {np.mean(f1_weighted):.4f} (± {np.std(f1_weighted):.4f})")

Baseline (k=5, euclidean, uniform): 0.3854 (± 0.0056)
Distance-weighted (k=5, euclidean): 0.3855 (± 0.0027)


In [ ]:
# ============================================================
# SECTION 4: Full k / distance-metric grid search (2-cluster
# branch, unclamped, single representative fold)
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full_2c[0]
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
metrics = ['euclidean', 'manhattan']

grid_results = []
for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train_f, y_train_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"metric={metric:10s} k={k:3d}  macro-F1={macro_f1:.4f}")

best = pd.DataFrame(grid_results).sort_values('macro_f1', ascending=False).iloc[0]
print(f"\nBest (pre-resampling): metric={best['metric']}, k={best['k']}, macro-F1={best['macro_f1']:.4f}")

metric=euclidean  k=  1  macro-F1=0.3701
metric=euclidean  k=  3  macro-F1=0.3773
metric=euclidean  k=  5  macro-F1=0.3821
metric=euclidean  k=  7  macro-F1=0.3807
metric=euclidean  k=  9  macro-F1=0.3788
metric=euclidean  k= 15  macro-F1=0.3799
metric=euclidean  k= 21  macro-F1=0.3780
metric=euclidean  k= 31  macro-F1=0.3703
metric=manhattan  k=  1  macro-F1=0.3879
metric=manhattan  k=  3  macro-F1=0.3948
metric=manhattan  k=  5  macro-F1=0.4051
metric=manhattan  k=  7  macro-F1=0.4088
metric=manhattan  k=  9  macro-F1=0.4090
metric=manhattan  k= 15  macro-F1=0.4091
metric=manhattan  k= 21  macro-F1=0.4064
metric=manhattan  k= 31  macro-F1=0.3944

Best (pre-resampling): metric=manhattan, k=15, macro-F1=0.4091


In [ ]:
# ============================================================
# SECTION 5: Feature selection — screen then confirm
# (2-cluster branch, unclamped)
# ============================================================
mi_scores = mutual_info_classif(X_train_f, y_train_f, random_state=42)
mi_ranking = pd.Series(mi_scores, index=X_train_f.columns).sort_values(ascending=False)

for n in [10, 20, 30, 40]:
    top_n = mi_ranking.head(n).index.tolist()
    model = KNeighborsClassifier(n_neighbors=int(best['k']), weights='distance', metric=best['metric'])
    model.fit(X_train_f[top_n], y_train_f)
    print(f"top {n:3d} features: macro-F1 = {f1_score(y_test_f, model.predict(X_test_f[top_n]), average='macro'):.4f}")

fold_data_selected_2c = []
fold_selected_features_2c = []
for X_train, X_test, y_train, y_test in fold_data_full_2c:
    mi = mutual_info_classif(X_train, y_train, random_state=42)
    ranking = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    top10 = ranking.head(10).index.tolist()
    fold_selected_features_2c.append(top10)
    fold_data_selected_2c.append((X_train[top10], X_test[top10], y_train, y_test))

f1_mi = [f1_score(y_test, KNeighborsClassifier(n_neighbors=int(best['k']), weights='distance', metric=best['metric']).fit(X_train, y_train).predict(X_test), average='macro')
         for X_train, X_test, y_train, y_test in fold_data_selected_2c]
print(f"\nConfirmed top-10 (5-fold): {np.mean(f1_mi):.4f} (± {np.std(f1_mi):.4f})")
print("Selected features per fold:", fold_selected_features_2c)

top  10 features: macro-F1 = 0.5409
top  20 features: macro-F1 = 0.4702
top  30 features: macro-F1 = 0.4256
top  40 features: macro-F1 = 0.4079

Confirmed top-10 (5-fold): 0.5377 (± 0.0079)
Selected features per fold: [['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dinpkt', 'dload', 'dpkts'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dpkts', 'dinpkt'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']]


In [ ]:
# ============================================================
# SECTION 6: Build the leak-safe pipeline for the 7-cluster
# branch (unclamped), then run its own baseline + k/metric grid
# ============================================================
fold_data_full_7c = []
for fold_num, (train_idx, test_idx) in enumerate(skf_7c.split(X_7c, y_7c)):
    X_train, X_test = X_7c.iloc[train_idx].copy(), X_7c.iloc[test_idx].copy()
    y_train, y_test = y_7c.iloc[train_idx], y_7c.iloc[test_idx]

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    proto_counts = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full_7c.append((X_train_enc, X_test_enc, y_train, y_test))
    print(f"Fold {fold_num+1}: prepared. X_train: {X_train_enc.shape}")

X_train_f7, X_test_f7, y_train_f7, y_test_f7 = fold_data_full_7c[0]
grid_results_7c = []
for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train_f7, y_train_f7)
        macro_f1 = f1_score(y_test_f7, model.predict(X_test_f7), average='macro')
        grid_results_7c.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"metric={metric:10s} k={k:3d}  macro-F1={macro_f1:.4f}")

best_7c = pd.DataFrame(grid_results_7c).sort_values('macro_f1', ascending=False).iloc[0]
print(f"\nBest (7-cluster, pre-resampling): metric={best_7c['metric']}, k={best_7c['k']}, macro-F1={best_7c['macro_f1']:.4f}")

Fold 1: prepared. X_train: (130196, 62)
Fold 2: prepared. X_train: (130196, 62)
Fold 3: prepared. X_train: (130196, 63)
Fold 4: prepared. X_train: (130196, 63)
Fold 5: prepared. X_train: (130196, 62)
metric=euclidean  k=  1  macro-F1=0.3952
metric=euclidean  k=  3  macro-F1=0.3989
metric=euclidean  k=  5  macro-F1=0.4078
metric=euclidean  k=  7  macro-F1=0.4127
metric=euclidean  k=  9  macro-F1=0.4135
metric=euclidean  k= 15  macro-F1=0.4086
metric=euclidean  k= 21  macro-F1=0.3987
metric=euclidean  k= 31  macro-F1=0.3949
metric=manhattan  k=  1  macro-F1=0.4156
metric=manhattan  k=  3  macro-F1=0.4198
metric=manhattan  k=  5  macro-F1=0.4339
metric=manhattan  k=  7  macro-F1=0.4380
metric=manhattan  k=  9  macro-F1=0.4338
metric=manhattan  k= 15  macro-F1=0.4308
metric=manhattan  k= 21  macro-F1=0.4270
metric=manhattan  k= 31  macro-F1=0.4233

Best (7-cluster, pre-resampling): metric=manhattan, k=7, macro-F1=0.4380


In [ ]:
# ============================================================
# SECTION 7: Feature selection for the 7-cluster branch
# ============================================================
mi_scores_7c = mutual_info_classif(X_train_f7, y_train_f7, random_state=42)
mi_ranking_7c = pd.Series(mi_scores_7c, index=X_train_f7.columns).sort_values(ascending=False)

fold_data_selected_7c = []
fold_selected_features_7c = []
for X_train, X_test, y_train, y_test in fold_data_full_7c:
    mi = mutual_info_classif(X_train, y_train, random_state=42)
    ranking = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    top10 = ranking.head(10).index.tolist()
    fold_selected_features_7c.append(top10)
    fold_data_selected_7c.append((X_train[top10], X_test[top10], y_train, y_test))

f1_mi_7c = [f1_score(y_test, KNeighborsClassifier(n_neighbors=int(best_7c['k']), weights='distance', metric=best_7c['metric']).fit(X_train, y_train).predict(X_test), average='macro')
            for X_train, X_test, y_train, y_test in fold_data_selected_7c]
print(f"Confirmed top-10 (7-cluster, 5-fold): {np.mean(f1_mi_7c):.4f} (± {np.std(f1_mi_7c):.4f})")
print("Selected features per fold:", fold_selected_features_7c)

Confirmed top-10 (7-cluster, 5-fold): 0.5271 (± 0.0051)
Selected features per fold: [['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dinpkt', 'dload'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sload', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']]


In [ ]:
# ============================================================
# SECTION 8: Head-to-head comparison, both branches
# ============================================================
print("2-cluster branch:", f"{np.mean(f1_mi):.4f} (± {np.std(f1_mi):.4f})")
print("7-cluster branch:", f"{np.mean(f1_mi_7c):.4f} (± {np.std(f1_mi_7c):.4f})")

t_stat, p_val = stats.ttest_rel(f1_mi, f1_mi_7c)
print(f"Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

2-cluster branch: 0.5377 (± 0.0079)
7-cluster branch: 0.5271 (± 0.0051)
Paired t-test: t=1.9820, p=0.1185


In [ ]:
print(best)

k                  15
metric      manhattan
macro_f1     0.409056
Name: 13, dtype: object


In [ ]:
# ============================================================
# Custom distance-weighting function: inverse-squared distance,
# tested against the standard inverse-distance (1/d) already
# adopted, on the final configuration
# ============================================================
def inverse_squared_weights(distances):
    # Handle exact zero distances (duplicate points) the same way
    # sklearn's built-in 'distance' option does internally — a
    # zero-distance neighbour gets full weight, others get zero,
    # rather than crashing on division by zero.
    with np.errstate(divide='ignore'):
        weights = 1.0 / (distances ** 2)
    inf_mask = np.isinf(weights)
    if np.any(inf_mask):
        for i, row_mask in enumerate(inf_mask):
            if row_mask.any():
                weights[i] = row_mask.astype(float)
    return weights

X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected_2c[0]
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train_f, y_train_f)

# Standard inverse-distance (already adopted)
model_std = KNeighborsClassifier(n_neighbors=7, weights='distance', metric=best['metric'])
model_std.fit(Xs, ys)
print(f"Inverse-distance (1/d):    macro-F1 = {f1_score(y_test_f, model_std.predict(X_test_f), average='macro'):.4f}")

# Custom inverse-squared distance
model_sq = KNeighborsClassifier(n_neighbors=7, weights=inverse_squared_weights, metric=best['metric'])
model_sq.fit(Xs, ys)
print(f"Inverse-squared (1/d²):   macro-F1 = {f1_score(y_test_f, model_sq.predict(X_test_f), average='macro'):.4f}")

Inverse-distance (1/d):    macro-F1 = 0.5405
Inverse-squared (1/d²):   macro-F1 = 0.5384


In [ ]:
# ============================================================
# SECTION A: Proper k re-validation on resampled data —
# broad candidate set, single source of truth for k from here on
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected_2c[0]
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train_f, y_train_f)

k_candidates = [3, 5, 7, 9, 15, 21]
k_results = []
for k in k_candidates:
    model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='manhattan')
    model.fit(Xs, ys)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    k_results.append({'k': k, 'macro_f1': macro_f1})
    print(f"k={k:3d}: macro-F1 = {macro_f1:.4f}")

FINAL_K = max(k_results, key=lambda r: r['macro_f1'])['k']
print(f"\nFINAL_K = {FINAL_K}")

k=  3: macro-F1 = 0.5302
k=  5: macro-F1 = 0.5349
k=  7: macro-F1 = 0.5405
k=  9: macro-F1 = 0.5282
k= 15: macro-F1 = 0.5352
k= 21: macro-F1 = 0.5405

FINAL_K = 7


In [ ]:
# ============================================================
# SECTION B: Distance-weighting function test, at FINAL_K
# ============================================================
def inverse_squared_weights(distances):
    with np.errstate(divide='ignore'):
        weights = 1.0 / (distances ** 2)
    inf_mask = np.isinf(weights)
    if np.any(inf_mask):
        for i, row_mask in enumerate(inf_mask):
            if row_mask.any():
                weights[i] = row_mask.astype(float)
    return weights

model_std = KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan')
model_std.fit(Xs, ys)
f1_std = f1_score(y_test_f, model_std.predict(X_test_f), average='macro')
print(f"Inverse-distance (1/d):  macro-F1 = {f1_std:.4f}")

model_sq = KNeighborsClassifier(n_neighbors=FINAL_K, weights=inverse_squared_weights, metric='manhattan')
model_sq.fit(Xs, ys)
f1_sq = f1_score(y_test_f, model_sq.predict(X_test_f), average='macro')
print(f"Inverse-squared (1/d\u00b2): macro-F1 = {f1_sq:.4f}")

FINAL_WEIGHTS = 'distance' if f1_std >= f1_sq else inverse_squared_weights
print(f"\nFINAL_WEIGHTS = {'distance (1/d)' if FINAL_WEIGHTS == 'distance' else 'inverse-squared (1/d²)'}")

Inverse-distance (1/d):  macro-F1 = 0.5405
Inverse-squared (1/d²): macro-F1 = 0.5384

FINAL_WEIGHTS = distance (1/d)


In [ ]:
# ============================================================
# SECTION C: Resampling strategy comparison, redone at FINAL_K
# and FINAL_WEIGHTS (5-fold, not single-fold — this is the one
# that determines the final adopted strategy)
# ============================================================
f1_flat, f1_smote, f1_ros, f1_tomek, f1_smote_tomek = [], [], [], [], []

for X_train, X_test, y_train, y_test in fold_data_selected_2c:
    target_flat = {cls: max(count, 5000) for cls, count in y_train.value_counts().items()}
    Xs_i, ys_i = SMOTE(sampling_strategy=target_flat, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_flat.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan').fit(Xs_i, ys_i).predict(X_test), average='macro'))

    target_capped_i = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xs_i, ys_i = SMOTE(sampling_strategy=target_capped_i, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_smote.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan').fit(Xs_i, ys_i).predict(X_test), average='macro'))

    Xr_i, yr_i = RandomOverSampler(sampling_strategy=target_capped_i, random_state=42).fit_resample(X_train, y_train)
    f1_ros.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan').fit(Xr_i, yr_i).predict(X_test), average='macro'))

    Xt_i, yt_i = TomekLinks().fit_resample(X_train, y_train)
    f1_tomek.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan').fit(Xt_i, yt_i).predict(X_test), average='macro'))

    smote_st_i = SMOTE(sampling_strategy=target_capped_i, random_state=42, k_neighbors=5)
    Xst_i, yst_i = SMOTETomek(smote=smote_st_i, random_state=42).fit_resample(X_train, y_train)
    f1_smote_tomek.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan').fit(Xst_i, yst_i).predict(X_test), average='macro'))

print(f"SMOTE, flat target:       {np.mean(f1_flat):.4f} (± {np.std(f1_flat):.4f})")
print(f"SMOTE, ratio-capped 5x:   {np.mean(f1_smote):.4f} (± {np.std(f1_smote):.4f})")
print(f"Random oversampling:      {np.mean(f1_ros):.4f} (± {np.std(f1_ros):.4f})")
print(f"Tomek links alone:        {np.mean(f1_tomek):.4f} (± {np.std(f1_tomek):.4f})")
print(f"SMOTE + Tomek:            {np.mean(f1_smote_tomek):.4f} (± {np.std(f1_smote_tomek):.4f})")

t_stat, p_val = stats.ttest_rel(f1_smote_tomek, f1_smote)
print(f"\nPaired t-test, SMOTE+Tomek vs SMOTE: t={t_stat:.4f}, p={p_val:.4f}")

SMOTE, flat target:       0.5288 (± 0.0078)
SMOTE, ratio-capped 5x:   0.5382 (± 0.0100)
Random oversampling:      0.5288 (± 0.0053)
Tomek links alone:        0.5391 (± 0.0065)
SMOTE + Tomek:            0.5426 (± 0.0089)

Paired t-test, SMOTE+Tomek vs SMOTE: t=2.7932, p=0.0492


In [ ]:
# ============================================================
# SECTION D: Scaling confirmation, at FINAL_K, FINAL_WEIGHTS,
# and plain ratio-capped SMOTE (pending Section C's result)
# ============================================================
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train_f, y_train_f)

model_mm = KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan')
model_mm.fit(Xs, ys)
print(f"Min-max: macro-F1 = {f1_score(y_test_f, model_mm.predict(X_test_f), average='macro'):.4f}")

scaler_z = StandardScaler()
Xs_z = scaler_z.fit_transform(Xs)
Xtest_z = scaler_z.transform(X_test_f)
model_z = KNeighborsClassifier(n_neighbors=FINAL_K, weights=FINAL_WEIGHTS, metric='manhattan')
model_z.fit(Xs_z, ys)
print(f"Z-score: macro-F1 = {f1_score(y_test_f, model_z.predict(Xtest_z), average='macro'):.4f}")

Min-max: macro-F1 = 0.5405
Z-score: macro-F1 = 0.5366


In [ ]:
# ============================================================
# SECTION 15: FINAL MODEL — full 5-fold evaluation of the
# fully corrected, fully re-verified configuration
# ============================================================
final_f1, final_acc, final_weighted = [], [], []
all_y_test, all_y_pred = [], []

for X_train, X_test, y_train, y_test in fold_data_selected_2c:
    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(Xst, yst)
    y_pred = model.predict(X_test)

    final_f1.append(f1_score(y_test, y_pred, average='macro'))
    final_acc.append(accuracy_score(y_test, y_pred))
    final_weighted.append(f1_score(y_test, y_pred, average='weighted'))
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"Mean macro-F1:    {np.mean(final_f1):.4f} (± {np.std(final_f1):.4f})")
print(f"Mean accuracy:    {np.mean(final_acc):.4f} (± {np.std(final_acc):.4f})")
print(f"Mean weighted-F1: {np.mean(final_weighted):.4f} (± {np.std(final_weighted):.4f})")
print()
print(classification_report(all_y_test, all_y_pred, digits=3))

Mean macro-F1:    0.5426 (± 0.0089)
Mean accuracy:    0.7673 (± 0.0024)
Mean weighted-F1: 0.7786 (± 0.0017)

              precision    recall  f1-score   support

           0      0.926     0.838     0.880     83358
           1      0.712     0.744     0.727      8609
           2      0.187     0.622     0.287      1503
           3      0.349     0.405     0.375      5019
           4      0.810     0.772     0.791     26927
           5      0.141     0.067     0.090      1577
           6      0.525     0.671     0.589     19470
           7      0.559     0.669     0.609       169
           8      0.379     0.424     0.400      1426
           9      0.732     0.628     0.676      2185

    accuracy                          0.767    150243
   macro avg      0.532     0.584     0.542    150243
weighted avg      0.798     0.767     0.779    150243



In [ ]:
# ============================================================
# SECTION 16: SMOTE's internal k_neighbors sensitivity, at the
# final configuration (k=7, Manhattan, distance-weighted,
# SMOTE+Tomek, 5x ratio cap)
# ============================================================
for smote_k in [3, 5, 7, 10]:
    f1s = []
    for X_train, X_test, y_train, y_test in fold_data_selected_2c:
        target_capped = {cls: min(count * 5, y_train.value_counts().max())
                          for cls, count in y_train.value_counts().items()}
        smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=smote_k)
        Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
        m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
        m.fit(Xst, yst)
        f1s.append(f1_score(y_test, m.predict(X_test), average='macro'))
    print(f"SMOTE k_neighbors={smote_k}: {np.mean(f1s):.4f} (± {np.std(f1s):.4f})")

SMOTE k_neighbors=3: 0.5366 (± 0.0133)
SMOTE k_neighbors=5: 0.5426 (± 0.0089)
SMOTE k_neighbors=7: 0.5406 (± 0.0066)
SMOTE k_neighbors=10: 0.5376 (± 0.0082)


In [ ]:
# ============================================================
# SECTION 17: Manhattan vs Euclidean, redundant final check —
# at the fully finalized post-resampling configuration
# ============================================================
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected_2c[0]
target_capped = {cls: min(count * 5, y_train_f.value_counts().max())
                  for cls, count in y_train_f.value_counts().items()}
smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train_f, y_train_f)

for metric in ['manhattan', 'euclidean']:
    m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric=metric)
    m.fit(Xst, yst)
    print(f"{metric}: macro-F1 = {f1_score(y_test_f, m.predict(X_test_f), average='macro'):.4f}")

manhattan: macro-F1 = 0.5415
euclidean: macro-F1 = 0.5352
